# ANN-for-SIMS Pipeline

Artificial Neural Network for SIMS Prediction — modular pipeline with:
- Configurable train/val/test split
- Strict no-leakage evaluation (fold-fitted scalers in CV)
- Keras-Tuner hyperparameter search
- Optional retrain on train+val
- Calibrated empirical prediction intervals
- Diagnostic plots & 3D surfaces
- New-data prediction with PI

## 1. Install Dependencies

In [ ]:
%pip install -q tensorflow keras-tuner scikit-learn numpy pandas matplotlib seaborn openpyxl joblib

## 2. Configure the Pipeline

Adjust any parameters below before running. The three split percentages must sum to 100.

In [ ]:
from annsims.config import PipelineConfig, SurfacePlotConfig

cfg = PipelineConfig(
    # --- Data split (must sum to 100) ---
    train_percent=60,
    val_percent=20,
    test_percent=20,

    # --- Column layout ---
    n_labels=7,         # number of label (non-input) columns
    n_inputs=9,         # number of input feature columns
    target_col=None,    # absolute column index of target, or None for auto
    sep="\t",           # delimiter for text files

    # --- Hardware ---
    disable_gpu=True,

    # --- Tuning / training ---
    tuner_trials=15,
    k_folds=15,
    random_seed=42,
    tuner_epochs=200,
    cv_epochs=200,
    final_epochs=200,

    # --- Optional retrain on train+val ---
    do_optional_retrain=True,

    # --- Prediction interval ---
    pi_calibration="val",   # \"val\" or \"oof\"
    pi_alpha=0.05,          # 95% PI => alpha=0.05

    # --- Export ---
    export_dir="optimized_model",
)

surface_cfg = SurfacePlotConfig(
    grid_n=35,
    range_mode="quantile",
    q_low=0.02,
    q_high=0.98,
    hold_mode="median_train",
    overlay_train_scatter=True,
    scatter_alpha=0.25,
    scatter_size=8,
    max_pairs=12,
    dpi=160,
    features_to_use=None,   # e.g. ["MnCO3", "FeCO3"] or None for all pairs
)

print(f"Config ready: {cfg.train_percent}/{cfg.val_percent}/{cfg.test_percent} split")

## 3. Run the Full Pipeline

This will:
1. Upload / load your dataset
2. Split into train / val / test
3. Run Keras-Tuner hyperparameter search
4. Cross-validate on TRAIN with fold-fitted scalers (no leakage)
5. Final training on TRAIN, validated on VAL
6. Optionally retrain on TRAIN+VAL
7. Generate diagnostic plots & 3D surfaces
8. Export all results to Excel/CSV + ZIP
9. Predict on new data (if uploaded) with calibrated PI

In [ ]:
from annsims.main import run_pipeline

run_pipeline(cfg, surface_cfg=surface_cfg)

## 4. Download Results

After the pipeline completes, results are in `optimized_model/`.

**In Colab:** find the ZIP files in the Files sidebar and right-click → Download.

Or run the cell below to trigger a download dialog:

In [ ]:
import os

try:
    from google.colab import files as colab_files
    if os.path.exists("training_results.zip"):
        colab_files.download("training_results.zip")
    if os.path.exists("new_data_predictions.zip"):
        colab_files.download("new_data_predictions.zip")
except ImportError:
    print("Not in Colab. Check ~/Downloads/ or the optimized_model/ directory.")